# Step 2: Preprocessing — Ticket Resolution System

This notebook prepares `tickets.csv` for fine-tuning DistilBERT (Step 3).

**What this notebook does:**
1. Mount Google Drive (so files persist across Colab sessions)
2. Load `tickets.csv`
3. Light text cleaning (no aggressive NLP preprocessing — DistilBERT wants near-raw text)
4. Encode category labels as integers
5. Check class balance and compute class weights
6. Tokenize with DistilBERT's tokenizer
7. Train / validation / test split (stratified)
8. Save everything to Drive for Step 3

**Before running:** upload `tickets.csv` to a folder in your Google Drive, e.g.
`MyDrive/TicketResolutionSystem/data/tickets.csv`, and update `DATA_PATH` below if needed.


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Install dependencies

In [ ]:
!pip install -q transformers datasets scikit-learn pandas


## 3. Set paths and load the dataset

Adjust `DATA_PATH` if you placed `tickets.csv` somewhere else in your Drive.

In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/TicketResolutionSystem")
DATA_PATH = PROJECT_DIR / "data" / "tickets.csv"
OUTPUT_DIR = PROJECT_DIR / "processed_data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()


## 4. Light text cleaning

DistilBERT's tokenizer is pretrained on natural text, so we avoid lowercasing,
stopword removal, or stemming — those would actually *hurt* performance here.
We only fix whitespace and strip leftover anonymization artifacts.

In [ ]:
import re

def clean_text(text: str) -> str:
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r"<name>|<company>|<email>|<phone>", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["description"] = df["description"].apply(clean_text)
df["resolution"] = df["resolution"].apply(clean_text)

# Drop any rows that became empty after cleaning
df = df[df["description"].str.len() > 0].reset_index(drop=True)
print("Shape after cleaning:", df.shape)


## 5. Encode labels

Neural networks need integer targets, not strings. We map each category
to an integer ID and save the mapping so predictions can be decoded later.

In [ ]:
categories = sorted(df["category"].unique())
label2id = {label: i for i, label in enumerate(categories)}
id2label = {i: label for label, i in label2id.items()}

df["label"] = df["category"].map(label2id)

print("Label mapping:", label2id)
df[["category", "label"]].drop_duplicates().sort_values("label")


## 6. Check class balance and compute class weights

We saw earlier that categories are imbalanced (Performance Issue has ~3x the
examples of Server Down). Rather than discarding data to force balance, we
compute class weights to pass into the loss function during training in
Step 3 — this tells the model to penalize mistakes on minority classes more.

In [ ]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

print("Class distribution:")
print(df["category"].value_counts())
print()

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array(sorted(label2id.values())),
    y=df["label"].values,
)
class_weights = class_weights.astype(np.float32)

print("\nComputed class weights (higher = rarer class, weighted more in loss):")
for label, weight in zip(categories, class_weights):
    print(f"  {label}: {weight:.4f}")


## 7. Train / Validation / Test split (stratified)

80% train / 10% validation / 10% test. Stratified by category so all three
splits keep the same proportional class balance as the full dataset —
important given the imbalance above.

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df, test_size=0.2, stratify=df["label"], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df["label"], random_state=42
)

print(f"Train: {len(train_df)}  |  Val: {len(val_df)}  |  Test: {len(test_df)}")
print()
print("Train category distribution:")
print(train_df["category"].value_counts(normalize=True).round(3))


## 8. Tokenize with DistilBERT's tokenizer

Converts text into token IDs + attention masks the model can process.
`max_length=256` covers most ticket descriptions without wasting compute
padding very short ones to an unnecessarily long fixed size.

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_LENGTH = 256

def tokenize_split(split_df):
    encodings = tokenizer(
        split_df["description"].tolist(),
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="np",
    )
    return {
        "input_ids": encodings["input_ids"],
        "attention_mask": encodings["attention_mask"],
        "labels": split_df["label"].values,
    }

train_encoded = tokenize_split(train_df)
val_encoded = tokenize_split(val_df)
test_encoded = tokenize_split(test_df)

print("Train input_ids shape:", train_encoded["input_ids"].shape)
print("Example token IDs (first 20):", train_encoded["input_ids"][0][:20])
print("Decoded back to text:", tokenizer.decode(train_encoded["input_ids"][0][:20]))


## 9. Save everything to Drive

These files are what Step 3 (training) will load. Saving as `.npz` (numpy
compressed) for the tokenized arrays, plus a JSON for the label mapping
and class weights, plus the raw split CSVs for reference/debugging.

In [ ]:
import json

# Tokenized arrays
np.savez(
    OUTPUT_DIR / "train_encoded.npz",
    input_ids=train_encoded["input_ids"],
    attention_mask=train_encoded["attention_mask"],
    labels=train_encoded["labels"],
)
np.savez(
    OUTPUT_DIR / "val_encoded.npz",
    input_ids=val_encoded["input_ids"],
    attention_mask=val_encoded["attention_mask"],
    labels=val_encoded["labels"],
)
np.savez(
    OUTPUT_DIR / "test_encoded.npz",
    input_ids=test_encoded["input_ids"],
    attention_mask=test_encoded["attention_mask"],
    labels=test_encoded["labels"],
)

# Label mapping + class weights + config, needed by Step 3
metadata = {
    "label2id": label2id,
    "id2label": {str(k): v for k, v in id2label.items()},
    "class_weights": class_weights.tolist(),
    "model_name": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "num_labels": len(categories),
}
with open(OUTPUT_DIR / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

# Raw splits (handy for error analysis / inspecting predictions later)
train_df.to_csv(OUTPUT_DIR / "train.csv", index=False)
val_df.to_csv(OUTPUT_DIR / "val.csv", index=False)
test_df.to_csv(OUTPUT_DIR / "test.csv", index=False)

print(f"Saved all preprocessing outputs to: {OUTPUT_DIR}")
print(list(OUTPUT_DIR.iterdir()))


## Done ✅

You now have, saved in `MyDrive/TicketResolutionSystem/processed_data/`:
- `train_encoded.npz`, `val_encoded.npz`, `test_encoded.npz` — tokenized, model-ready data
- `metadata.json` — label mapping, class weights, tokenizer config
- `train.csv`, `val.csv`, `test.csv` — raw splits for reference

**Next: Step 3 — fine-tune DistilBERT** using these files. Bring this notebook's
`OUTPUT_DIR` path into the training notebook to load everything back in.
